# miniLLM 49M — reprise du pré-entraînement

Plan raccourci : 50 200 itérations (≈ 1,02 époque), schedule WSD (LR stable à 6e-4 jusqu'à l'it. 42 670, puis décroissance jusqu'à 6e-5). Reprise automatique depuis le dernier checkpoint du Drive.

**Utilisation :** Exécution → Modifier le type d'exécution → GPU (T4), puis **Tout exécuter**. Durée ≈ 8 h.

In [ ]:
# ── miniLLM 49M · reprise du pré-entraînement (plan raccourci, 1,02 époque) ──
# Il suffit de lancer cette cellule (ou « Tout exécuter »). Rien à coller ni à modifier.
import os, sys, glob, shutil
from google.colab import drive
drive.mount("/content/drive")

# 1) Dossier du projet sur le Drive (cherche aussi un raccourci / un Drive partagé)
cands = (["/content/drive/MyDrive/MiniLLM_v2"]
         + glob.glob("/content/drive/MyDrive/*/MiniLLM_v2")
         + glob.glob("/content/drive/Shareddrives/*/MiniLLM_v2"))
BASE = next((c for c in cands if os.path.isdir(f"{c}/data/pretrain")), None)
if BASE is None:
    raise SystemExit("Dossier MiniLLM_v2 introuvable dans ce Drive. S'il t'a été partagé : Drive > Partagés > "
                     "appui long sur MiniLLM_v2 > « Ajouter un raccourci » > Mon Drive, puis relance.")
print("Projet :", BASE)

PT_OUT, CODE_DIR = f"{BASE}/checkpoints/pretrain", "/content/minillm_v2"
MODEL_SIZE, PT_LR, PT_BATCH, TOKENS_PER_STEP = "49M", 6e-4, 16, 65_536
PT_ITERS, DECAY_FRAC = 50_200, 0.15      # 50 200 it. = 1,02 époque ; décroissance à partir de l'it. 42 670

# 2) Code du projet
!pip -q install tokenizers datasets
if os.path.exists(f"{CODE_DIR}/.git"):
    !git -C {CODE_DIR} pull --ff-only
else:
    !git clone https://github.com/bono-p/minillm_v2.git {CODE_DIR}
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)

import torch
N_GPU = torch.cuda.device_count()
if N_GPU == 0:
    raise SystemExit("Pas de GPU : Exécution > Modifier le type d'exécution > T4 GPU, puis relance.")
PT_ACCUM = max(1, TOKENS_PER_STEP // (PT_BATCH * 512 * N_GPU))

# 3) Garde-fou : il FAUT un checkpoint de reprise (sinon l'entraînement repartirait de zéro)
from checkpoint import latest_checkpoint, load_checkpoint
last = latest_checkpoint(PT_OUT)
if not last:
    raise SystemExit(f"Aucun checkpoint dans {PT_OUT} : arrêt pour ne pas repartir de zéro.")
ck = load_checkpoint(last)
print(f"Reprise prévue : {os.path.basename(last)} | it={ck['iter']} | best_val_loss={ck['best_val_loss']:.4f}")
if int(ck["iter"]) < 40_000:
    raise SystemExit("Le dernier checkpoint est antérieur à l'it. 40 000 : ce n'est pas le bon dossier. Arrêt.")
del ck

# 4) Données sur le disque local de Colab (plus rapide que Drive)
SRC, DST = f"{BASE}/data/pretrain", "/content/minillm_local/pretrain"
if not os.path.exists(f"{DST}/meta.json"):
    print("Copie des données sur le disque local (quelques minutes)…")
    os.makedirs("/content/minillm_local", exist_ok=True)
    shutil.copytree(SRC, DST)
shutil.copy(f"{BASE}/data/tokenizer.json", "/content/minillm_local/tokenizer.json")

# 5) Entraînement (reprise automatique ; si la session est coupée, relance la cellule)
args = (f"--mode pretrain --data_dir {DST} --out_dir {PT_OUT} --model_size {MODEL_SIZE} "
        f"--seq_len 512 --batch_size {PT_BATCH} --grad_accum {PT_ACCUM} --lr {PT_LR} --min_lr {PT_LR/10} "
        f"--max_iters {PT_ITERS} --schedule wsd --decay_frac {DECAY_FRAC} "
        f"--warmup_iters 300 --eval_every 500 --save_every 500 --eval_iters 50 --max_minutes 0")
if N_GPU > 1:
    !torchrun --standalone --nproc_per_node={N_GPU} train.py {args}
else:
    !python train.py {args}
